# Entity Resolution - exploration notebook

**This notebook is for exploration only.** Anything stable belongs in `src/`;
anything heavy must be runnable from `scripts/`. Nothing here is needed to
reproduce the pipeline.

What it answers:

1. Does the normalization preserve multilingual scripts? (Devanagari / Kannada)
2. What does the raw data look like? (counts, missingness)
3. What does the ground truth look like? (matches per S1, zero-match entities)
4. **What is the recall ceiling of exact-normalized-name blocking?** This is the
   number that decides what to build next.

Run the pipeline first (`prepare_data.py`, `build_indexes.py`,
`generate_candidates.py`, `evaluate_blocking.py`); this notebook reads the raw
TSVs and the ground truth.

## 0. Setup

In [ ]:
import sys, time
from pathlib import Path

import numpy as np
import pandas as pd

# Locate the repo root whether the notebook runs from notebooks/ or the root.
REPO = Path.cwd()
if not (REPO / "src").is_dir():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from src.data_loader import load_config, raw_path, iter_tsv, load_ground_truth
from src.normalization import Normalizer
from src.utils import decode_entity_ids

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

config = load_config()
DATA = config["resolved"]["data_root"]
norm = Normalizer.from_config(config)
print("repo:", REPO)
print("data:", DATA)
print("normalization:", norm.unicode_form, "| fold_latin_accents =", norm.fold_latin_accents)

## 1. Normalization spot checks

The load-bearing requirement: **combining marks must survive**. Indian records
appear in Devanagari, Kannada and Bengali, where the vowel signs and the virama
*are* combining marks. A naive NFD-then-strip-accents routine is right for French
and destroys Indic text, making every Indian business name collide with every
other.

In [ ]:
cases = [
    "Orelee's Barbershop",
    "Caf\u00e9 B\u00e9que",
    "B+ Retail Inc",
    "Pvt. EFS Print Ventures Ltd.",
    "heassociates.com",
    "FOUNDATION EXCEL AGENCY PRIVATE  LIMITED",
    "\u0930\u093e\u092e \u092e\u093e\u0930\u094d\u0915\u0947\u091f\u093f\u0902\u0917",  # Devanagari
    "\u0cb6\u0cbf\u0cb5\u0cb6\u0c95\u0ccd\u0ca4\u0cbf",                                # Kannada
    "H.No.16-11-23/37/A, 2Nd Floor",
]
for raw in cases:
    print(f"{raw[:44]:<46} -> {norm.name(raw)[:44]}")

In [ ]:
# Guard the property that matters: Indic text must come out unchanged.
indic = [
    "\u0930\u093e\u092e \u092e\u093e\u0930\u094d\u0915\u0947\u091f\u093f\u0902\u0917",
    "\u0915\u0902\u0938\u094d\u091f\u094d\u0930\u0915\u094d\u0936\u0902\u0938",
    "\u0cb6\u0cbf\u0cb5\u0cb6\u0c95\u0ccd\u0ca4\u0cbf \u0cb5\u0cbf\u0ca6\u0ccd\u0caf\u0cbe\u0cb2\u0caf",
]
for text in indic:
    out = norm.name(text)
    assert out == text, f"normalization altered Indic text: {text!r} -> {out!r}"
print("OK:", len(indic), "Indic strings pass through with combining marks intact")

# And the Latin behaviour we do want:
assert norm.name("Caf\u00e9") == "cafe"
assert norm.name("L\u00e9arning") == "learning"
print("OK: Latin accents folded")

## 2. Raw data profile

Streamed in chunks - the 10.3M-row sources are never fully resident.

In [ ]:
def profile(path, name, chunksize=1_000_000):
    rows = 0
    missing_name = missing_addr = 0
    ids = []
    for chunk in iter_tsv(
        path, usecols=["entity_id", "business_name", "business_address"], chunksize=chunksize
    ):
        rows += len(chunk)
        missing_name += int((chunk.business_name.str.strip() == "").sum())
        missing_addr += int((chunk.business_address.str.strip() == "").sum())
        ids.append(chunk.entity_id)
    all_ids = pd.concat(ids)
    return {
        "source": name,
        "rows": rows,
        "unique_ids": int(all_ids.nunique()),
        "missing_name": missing_name,
        "missing_address": missing_addr,
        "missing_address_pct": round(100 * missing_addr / rows, 2) if rows else 0.0,
    }


profile_table = pd.DataFrame(
    [profile(raw_path(config, "train", s), s) for s in ("source1", "source2", "source3")]
)
profile_table

**Address cannot be the primary blocking signal** - ~3.4% of target records have
no address at all (168,967 in S2 and 175,916 in S3). Any address-only blocker
would make ~170k records unreachable.

In [ ]:
# Unique normalized names - this drives index sizing.
name_counts = {}
for src in ("source1", "source2", "source3"):
    seen = set()
    for chunk in iter_tsv(
        raw_path(config, "train", src), usecols=["business_name"], chunksize=1_000_000
    ):
        seen.update(norm.series(chunk["business_name"], "name").unique().tolist())
    name_counts[src] = len(seen)
    print(f"{src}: {len(seen):,} unique normalized names")
name_counts

## 3. Ground truth profile

In [ ]:
gt = load_ground_truth(config, log=None)
print(gt.describe())

lengths = gt.lengths()
print()
print("matches per S1 (distribution):")
print(pd.Series(lengths).value_counts().sort_index().head(15).to_string())

### Zero-match entities

123,247 S1 entities have an empty ground-truth list. Under a per-entity
macro-averaged F0.5 that matters a lot: predicting *anything* for such an entity
drives its individual score to 0, and one spurious match is enough to do it.

In [ ]:
n_zero = int((lengths == 0).sum())
print(f"S1 with zero true matches : {n_zero:,} ({100 * n_zero / len(lengths):.2f}%)")
print(f"true matches total        : {gt.n_matches:,}")

# Per-source split of true pairs (the source is packed into the id code).
for code, prefix in ((2, "S2"), (3, "S3")):
    print(f"  true pairs from {prefix}: {int(((gt.codes // 10**10) == code).sum()):,}")

## 4. Exact-normalized-name blocking ceiling

The decisive experiment. For a sample of S1 entities, compare the normalized name
of the S1 record against the normalized names of its **true** matches. If they are
not string-identical, no exact-name blocker can ever retrieve that pair.

This is a ceiling: it upper-bounds the current blocker and any tuning of it, so it
tells us whether to tune the blocker or replace it.

In [ ]:
SAMPLE_SIZE = 100_000
SEED = 42

sample = pd.Series(gt.entity_ids).sample(n=SAMPLE_SIZE, random_state=SEED)
needed_by_s1 = {s1: gt.matches(s1) for s1 in sample}
all_needed_codes = np.unique(np.concatenate([v for v in needed_by_s1.values() if len(v)]))
needed_ids = set(decode_entity_ids(all_needed_codes).tolist())
print(f"sampled S1: {len(needed_by_s1):,} | distinct target ids needed: {len(needed_ids):,}")

In [ ]:
# Load only the names we need, streaming each source once.
needed_s1 = set(needed_by_s1.keys())

s1_names = {}
for chunk in iter_tsv(
    raw_path(config, "train", "source1"),
    usecols=["entity_id", "business_name"],
    chunksize=1_000_000,
):
    hit = chunk[chunk.entity_id.isin(needed_s1)]
    for entity_id, name in zip(hit.entity_id, hit.business_name):
        s1_names[entity_id] = norm.name(name)

target_names = {}
for src in ("source2", "source3"):
    for chunk in iter_tsv(
        raw_path(config, "train", src),
        usecols=["entity_id", "business_name"],
        chunksize=1_000_000,
    ):
        hit = chunk[chunk.entity_id.isin(needed_ids)]
        for entity_id, name in zip(hit.entity_id, hit.business_name):
            target_names[entity_id] = norm.name(name)

print(f"loaded {len(s1_names):,} S1 names and {len(target_names):,} target names")

In [ ]:
total = norm_hits = key_hits = 0
s1_with_matches = s1_full = s1_partial = 0

for s1_id, codes in needed_by_s1.items():
    true_ids = decode_entity_ids(codes).tolist()
    if not true_ids:
        continue
    s1_with_matches += 1
    a = s1_names.get(s1_id, "")
    hits = 0
    for tid in true_ids:
        total += 1
        b = target_names.get(tid, "")
        if a and b and a == b:
            norm_hits += 1
            hits += 1
        elif a and b and a.replace(" ", "") == b.replace(" ", ""):
            key_hits += 1
    if hits == len(true_ids):
        s1_full += 1
    if hits:
        s1_partial += 1

summary = pd.Series(
    {
        "true pairs in sample": total,
        "pair recall (name_norm)": f"{100 * norm_hits / total:.2f}%",
        "pair recall (name_norm + name_key)": f"{100 * (norm_hits + key_hits) / total:.2f}%",
        "S1 with >=1 true match": s1_with_matches,
        "S1 FULL recall (all matches found)": f"{100 * s1_full / s1_with_matches:.2f}%",
        "S1 partial recall (>=1 found)": f"{100 * s1_partial / s1_with_matches:.2f}%",
    },
    name="value",
)
summary

## 5. What this means

The ceiling is low, and it is the most important number for planning:

* **~26% pair recall.** Three quarters of true matches share no exact normalized
  name with their reference entity.
* **~3.7% of S1 entities have all their matches retrieved.** Under a per-entity
  macro F0.5 an entity missing even one match has a hard ceiling on its own score,
  so this - not pair recall - is the number that tracks the leaderboard.
* **~60% of entities have at least one match retrieved**, which is why the
  candidate set looks healthy while being far from usable.

Exact-name blocking is a foundation, not a solution. What moves recall:

1. **Token / inverted-index blocking** - matches on shared tokens instead of the
   whole string. Cheapest large win; the generic index + union machinery in
   `src/blocking.py` already supports it (`BLOCKER_TOKEN` is registered).
2. **Character n-gram retrieval** - absorbs typos and transliteration variance.
3. **Multilingual dense retrieval** - the noise includes substitutions no lexical
   signal survives (`Delta Tetlecommunication`). Needs a multilingual encoder;
   GPU when available, CPU otherwise.

Precision needs work too: even where names match exactly, only ~7.7% of those
pairs are true matches - distinct businesses genuinely share names. With
`beta = 0.5` a false positive costs 4x a false negative, so the matcher must be
precision-oriented.

**Re-run `scripts/evaluate_blocking.py` after every blocker change.** The point of
measuring the ceiling first is to avoid tuning a model against a candidate set
that is missing most of the answers.